# Data Cleaning
Use prem data for exploratory data analysis

In [2]:
import pandas as pd
from pathlib import Path


def load_prem_data(year) -> pd.DataFrame:
    """Load yyyy.csv as a pandas DataFrame."""
    csv_file = Path(
        "/accounts/masters/gautierep/footy_prediction/footy-prediction"
    ) / "data" / "raw" / "football_data" / f"{year}.csv"

    if not csv_file.exists():
        raise FileNotFoundError(f"Could not find {csv_file}")

    return pd.read_csv(csv_file, encoding="latin-1")


df_2526 = load_prem_data(2526)
df_2526.head()

,ï»¿Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,15/08/2025,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,16/08/2025,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,16/08/2025,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,16/08/2025,15:00,Sunderland,West Ham,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,16/08/2025,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92


In [ ]:
df_2526["date"] = pd.to_datetime(
    df_2526["Date"],
    dayfirst=True
)

df_2526 = df_2526.sort_values("date")

df_2526["home_points"] = df_2526["FTR"].map({
    "H": 3,
    "D": 1,
    "A": 0
})

df_2526["away_points"] = df_2526["FTR"].map({
    "H": 0,
    "D": 1,
    "A": 3
})

In [ ]:
import numpy as np


def build_prematch_features(matches: pd.DataFrame) -> pd.DataFrame:
    """Create leakage-safe pre-match features from football match data."""
    data = matches.copy()
    data = data.rename(columns={"ï»¿Div": "Div"})
    data["kickoff"] = pd.to_datetime(
        data["Date"].astype(str) + " " + data["Time"].fillna("00:00").astype(str),
        dayfirst=True,
        errors="coerce",
    )
    data = data.sort_values("kickoff").reset_index(drop=True)
    data["match_id"] = np.arange(len(data))

    # Market probabilities from average 1X2 odds.
    data["market_home_prob"] = 1 / data["AvgH"]
    data["market_draw_prob"] = 1 / data["AvgD"]
    data["market_away_prob"] = 1 / data["AvgA"]
    overround = data[
        ["market_home_prob", "market_draw_prob", "market_away_prob"]
    ].sum(axis=1)
    for outcome in ["home", "draw", "away"]:
        data[f"market_{outcome}_prob_fair"] = (
            data[f"market_{outcome}_prob"] / overround
        )

    # Put each match into a team-level history table.
    home_history = pd.DataFrame({
        "match_id": data["match_id"],
        "kickoff": data["kickoff"],
        "team": data["HomeTeam"],
        "opponent": data["AwayTeam"],
        "venue": "home",
        "goals_for": data["FTHG"],
        "goals_against": data["FTAG"],
        "points": data["FTR"].map({"H": 3, "D": 1, "A": 0}),
        "shots_for": data["HS"],
        "shots_on_target_for": data["HST"],
    })
    away_history = pd.DataFrame({
        "match_id": data["match_id"],
        "kickoff": data["kickoff"],
        "team": data["AwayTeam"],
        "opponent": data["HomeTeam"],
        "venue": "away",
        "goals_for": data["FTAG"],
        "goals_against": data["FTHG"],
        "points": data["FTR"].map({"H": 0, "D": 1, "A": 3}),
        "shots_for": data["AS"],
        "shots_on_target_for": data["AST"],
    })
    history = pd.concat([home_history, away_history], ignore_index=True)
    history = history.sort_values(["team", "kickoff", "match_id"])

    rolling_columns = [
        "points",
        "goals_for",
        "goals_against",
        "shots_for",
        "shots_on_target_for",
    ]
    feature_frames = []

    for team, team_history in history.groupby("team", sort=False):
        team_history = team_history.copy()
        team_history["rest_days"] = (
            team_history["kickoff"].diff().dt.total_seconds() / 86400
        )
        for window in [5, 10]:
            for column in rolling_columns:
                feature_name = f"{column}_last_{window}"
                team_history[feature_name] = (
                    team_history[column]
                    .shift(1)
                    .rolling(window, min_periods=1)
                    .mean()
                )
        feature_frames.append(team_history)

    history_features = pd.concat(feature_frames)
    feature_columns = [
        "match_id",
        "venue",
        "rest_days",
        *[
            f"{column}_last_{window}"
            for window in [5, 10]
            for column in rolling_columns
        ],
    ]

    home_features = history_features.loc[
        history_features["venue"] == "home", feature_columns
    ].rename(columns={
        column: f"home_{column}"
        for column in feature_columns
        if column not in ["match_id", "venue"]
    })
    away_features = history_features.loc[
        history_features["venue"] == "away", feature_columns
    ].rename(columns={
        column: f"away_{column}"
        for column in feature_columns
        if column not in ["match_id", "venue"]
    })

    features = data.merge(home_features.drop(columns="venue"), on="match_id")
    features = features.merge(away_features.drop(columns="venue"), on="match_id")
    features["points_difference_last_5"] = (
        features["home_points_last_5"] - features["away_points_last_5"]
    )
    features["goals_for_difference_last_5"] = (
        features["home_goals_for_last_5"] - features["away_goals_for_last_5"]
    )
    features["goals_against_difference_last_5"] = (
        features["home_goals_against_last_5"]
        - features["away_goals_against_last_5"]
    )

    return features


prematch_2526 = build_prematch_features(df_2526)
prematch_2526[[
    "HomeTeam",
    "AwayTeam",
    "FTR",
    "home_points_last_5",
    "away_points_last_5",
    "points_difference_last_5",
    "market_home_prob_fair",
    "market_draw_prob_fair",
    "market_away_prob_fair",
]].head()